# Stage 1: nested variants B and D (through my runner)

Same benchmark, model size, data order, batch size, epochs, clipping and metrics as the Flat / LLRD control.
All three arms here use the team's ResNet class (`reduced_resnet18_blocks`, same architecture and, per the tests,
the same initial weights as my ResNet) and the team's `NestedLearner`, split into three levels:
`[stem, layer1] | [layer2, layer3] | [layer4, fc]`, `slow_side: input` (input-side level slowest).

- **Flat (`flat_nl`)**: variant B with all three ratios at 1, i.e. plain SGD through the same code path.
- **B (`nested_b`)**: per-level learning rates lr/16, lr/4, lr (input side first). With this split it is the same
  thing as my Flat + LLRD with `slow_side: input` (old label `llrd_fast_deep`), so section 2 is a regression check.
- **D (`nested_d_lam…`), one reading of the plan's "nested objectives"**: every level trains on the data loss;
  levels step every 16 / 4 / 1 minibatches (input side first) on their averaged gradients, all at the base lr; and each
  slower level also gets λ × the gradient of the faster neighbour's squared gradient norm (double backward).
  λ = 0 is variant A (periodic stepping) at the same lr.

Clipping: every arm clips each minibatch's new gradient to norm 1.0; D's accumulated gradients are not clipped again.
Logged per task: optimizer updates per level, how far each level's weights moved, D's penalty-to-data gradient ratio,
minutes, peak GPU memory.

Order: 0 tests → 1 smoke → 2 regression + B lr grid → 3 D lr grid, then λ grid → 4 control (test set, 3 seeds).
Every section is resumable: finished runs are skipped.

In [ ]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # <-- EDIT ONCE
BRANCH = "nested-b-d"   # the branch with the code this notebook needs (set back to "main" after merging)

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", url, CODE], check=True)
# fetch + checkout + pull, so a clone left on another branch earlier in the session switches over
subprocess.run(["git", "-C", CODE, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", CODE, "pull", "-q", "origin", BRANCH], check=True)
print(BRANCH, subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
_MINE = {"fyp_infra", "stage1_runner", "stage1_analysis", "stage1_drift_diagnostic",
         "stage1_structural", "nested_builder"}
for _m in list(sys.modules):
    if _m in _MINE or _m == "src" or _m.startswith("src."):   # src.* = the team's nested code
        sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop for training: a CPU run is ~12x slower and its timings are useless for budgeting.
REQUIRE_GPU = True
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")

In [ ]:
import numpy as np, pandas as pd

def table(configs):
    """One row per config: accuracy metrics, cost, and per-level means over tasks (nested arms)."""
    rows = []
    for cfg in configs:
        base = {"arm": sr.arm_label(cfg), "lr": cfg["lr"], "seed": cfg["seed"]}
        r = load_result(cfg, paths)
        if r is None:
            rows.append({**base, "status": "not finished"}); continue
        m = r["metrics"]
        row = {**base, "run_id": r["run_id"], "avg_incremental_acc": m["avg_incremental_acc"],
               "final_avg_acc": m["final_avg_acc"], "avg_forgetting": m["avg_forgetting"],
               # mean of the diagonal: accuracy on each task right after learning it (plasticity check)
               "acc_just_learned": float(np.mean(np.diag(m["acc_matrix"]))),
               "steps": m["optimizer_steps"], "n_params": m["n_params"],
               "minutes": r["wall_clock_s"] / 60, "peak_gpu_mb": m.get("peak_gpu_mem_mb")}
        log = m.get("level_log_per_exp") or []
        for k in (log[0] if log else {}):
            if k.startswith(("updates/", "rel_displacement/", "penalty_ratio/")):
                row[k] = float(np.mean([x[k] for x in log]))
        rows.append(row)
    return pd.DataFrame(rows)

## 0. Tests (tiny CPU configs, a few minutes)
Run after every `git pull`. Stop here if anything fails.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"], cwd=CODE, capture_output=True, text=True,
                   env={**os.environ, "WANDB_MODE": "disabled", "CUDA_VISIBLE_DEVICES": ""})   # CPU: deterministic
print(r.stdout[-6000:]); print(r.stderr[-3000:])
assert r.returncode == 0, "tests failed: read the output above before running anything else"

## 1. Smoke test (2 experiences × 50 classes, 1 epoch, validation split)

Checks that each arm runs on the GPU with Avalanche 0.6.0 (the team's code was written for 0.5.0), and measures cost.
One epoch over 2 experiences sees the same training images as one epoch of the 20-task setup, so a full
10-epoch run trains for about **10 × `train_min`** (evaluation time comes on top).
`penalty_ratio/<level>` = D's penalty gradient size / data gradient size for that level (input for setting the λ grid).

In [ ]:
SMOKE = [sr.arm_cfg(name, stage="smoke_nested", n_experiences=2, epochs=1, lr=0.1, eval_split="val")
         for name in ["flat", "nested_b", "nested_d"]]
out = run_sweep(SMOKE, sr.run_experiment, paths, force=True)   # force: always re-time after code changes
smoke = table(SMOKE)
smoke["train_min"] = [sum(out[run_id(c)]["metrics"]["train_minutes_per_exp"]) if out[run_id(c)] else None
                      for c in SMOKE]
smoke["est_full_run_train_min"] = 10 * smoke["train_min"]
display(smoke.T)

## 2. Regression check, then B's learning-rate grid (validation, seed 0)

**Regression check (criterion fixed before running):** B at lr 0.5 must match the old-path LLRD run with the same
settings (`llrd_fast_deep`, lr 0.5, from `stage1_lr_grid`) to within **0.5 points** on average incremental accuracy and
on accuracy on the just-learned task. If the init test passed, the only difference is GPU non-determinism.
If it fails, stop: the nested path is not reproducing the control.

**B's grid:** the old grid for this arm stopped at its best value, 0.5, and the control then used 0.6 without
validating it. Here B gets 0.5 / 0.6 / 0.7 / 0.9. Selection rule as before: highest `avg_incremental_acc`;
if the best is at the top of the grid, extend it. The old-path points below 0.5 (0.03-0.3) are shown for context
(they are only comparable if the regression check passes).

In [ ]:
EPOCHS = 10

def val_cfg(name, lr, **kw):
    return sr.arm_cfg(name, stage="stage1_nested_lr_grid", epochs=EPOCHS, seed=0, lr=lr, eval_split="val", **kw)

OLD_B = {lr: sr.make_cfg(stage="stage1_lr_grid", strategy="naive", epochs=EPOCHS, seed=0, lr=lr, eval_split="val",
                         lr_schedule="llrd", level_ratios=(1, 4, 16), level_order="fast_deep")
         for lr in [0.03, 0.1, 0.2, 0.3, 0.5]}
B_GRID = [0.5, 0.6, 0.7, 0.9]     # 0.5 first: it is the regression run
B_CONFIGS = [val_cfg("nested_b", lr) for lr in B_GRID]
for c in B_CONFIGS: print(run_id(c), sr.arm_label(c), c["lr"])

In [ ]:
out = run_sweep(B_CONFIGS, sr.run_experiment, paths)

In [ ]:
old, new = load_result(OLD_B[0.5], paths), load_result(B_CONFIGS[0], paths)
assert old is not None, "old llrd_fast_deep lr 0.5 val run not found on Drive"
if new is not None:
    reg = table([OLD_B[0.5], B_CONFIGS[0]])
    display(reg[["arm", "lr", "run_id", "avg_incremental_acc", "acc_just_learned", "final_avg_acc", "avg_forgetting", "minutes"]])
    d_inc = abs(reg["avg_incremental_acc"].iloc[0] - reg["avg_incremental_acc"].iloc[1])
    d_jl = abs(reg["acc_just_learned"].iloc[0] - reg["acc_just_learned"].iloc[1])
    A_old, A_new = np.array(old["metrics"]["acc_matrix"]), np.array(new["metrics"]["acc_matrix"])
    print(f"|diff| avg incremental {d_inc:.2f}, just-learned {d_jl:.2f}; acc-matrix max |diff| {np.abs(A_old - A_new).max():.2f}")
    print("REGRESSION CHECK", "PASSED" if max(d_inc, d_jl) < 0.5 else "FAILED: stop and investigate")

In [ ]:
bdf = table(list(OLD_B.values()) + B_CONFIGS)
display(bdf[["arm", "lr", "avg_incremental_acc", "acc_just_learned", "final_avg_acc", "minutes"]])
done = bdf[bdf["arm"] == "nested_b"].dropna(subset=["avg_incremental_acc"])
if len(done):
    best = done.loc[done["avg_incremental_acc"].idxmax()]
    print(f"best B lr on its grid: {best['lr']} (avg incremental {best['avg_incremental_acc']:.2f})")
    if best["lr"] == max(B_GRID): print("WARNING: best is at the top of the grid; extend B_GRID before choosing.")

## 3. D: learning-rate grid (λ = 0.1), then λ grid at the chosen lr (validation, seed 0)

Same selection rule. The λ grid is searched at D's best lr (one parameter at a time, not a full lr × λ grid,
to fit the budget). λ = 0 equals variant A at that lr: it shows whether the penalty adds anything to periodic stepping.
If the smoke test's `penalty_ratio` is far from 0.01-1 at λ = 0.1, re-centre `LAM_GRID` before running it (and note why).

In [ ]:
D_LR_GRID = [0.03, 0.1, 0.2, 0.3, 0.5]
D_LR_CONFIGS = [val_cfg("nested_d", lr) for lr in D_LR_GRID]   # surprise_weight 0.1 from nested_d.yaml
for c in D_LR_CONFIGS: print(run_id(c), sr.arm_label(c), c["lr"])
out = run_sweep(D_LR_CONFIGS, sr.run_experiment, paths)

In [ ]:
ddf = table(D_LR_CONFIGS)
display(ddf.drop(columns=["seed", "run_id"], errors="ignore"))
done = ddf.dropna(subset=["avg_incremental_acc"]) if "avg_incremental_acc" in ddf else ddf.iloc[:0]
if len(done):
    best = done.loc[done["avg_incremental_acc"].idxmax()]
    print(f"best D lr: {best['lr']} (avg incremental {best['avg_incremental_acc']:.2f})")
    if best["lr"] in (min(D_LR_GRID), max(D_LR_GRID)): print("WARNING: best lr is at the edge of the grid; extend it first.")

In [ ]:
D_LR = None                          # <- fill in from the table above
# Re-centred after the smoke test (lr 0.1, lambda 0.1): penalty/data gradient ratio was 3.9 (stem+layer1) and
# 1.2 (layer2+layer3), i.e. the penalty dominated. This grid spans ratio ~0.1 to ~4 on the slow level.
# 0.1 at D_LR is already in the lr grid (same config -> reused). 0 = variant A at the same lr.
LAM_GRID = [0.0, 0.003, 0.01, 0.03, 0.1]
if D_LR is None:
    print("set D_LR first")
else:
    D_LAM_CONFIGS = [val_cfg("nested_d", D_LR, rule_kwargs=dict(surprise_weight=lam)) for lam in LAM_GRID]
    for c in D_LAM_CONFIGS: print(run_id(c), sr.arm_label(c), c["lr"])
    out = run_sweep(D_LAM_CONFIGS, sr.run_experiment, paths)
    display(table(D_LAM_CONFIGS).drop(columns=["seed", "run_id"], errors="ignore"))

## 4. Control: Flat, B, D on the test set (3 seeds, checkpoints)

Fill `LR` and `D_LAM` from sections 2-3 (Flat uses its tuned lr 0.1 from `stage1_lr_grid`). An arm still at `None` is
skipped. Checkpoints go to `MyDrive/fyp_nested_cl/checkpoints/` for `stage1_analysis`, `stage1_drift_diagnostic` and
`stage1_structural` (all CPU). The old-path control runs (Flat lr 0.1, `llrd_fast_deep` lr 0.6) are shown for comparison.
The summary **asserts** equal optimizer-step counts (minibatch backward passes) and parameter counts across all rows.

In [ ]:
SEEDS = [0, 1, 2]
LR = {"flat": 0.1, "nested_b": None, "nested_d": None}
D_LAM = None
CONTROL = []
for s in SEEDS:                       # seed outer: each seed completes every arm before the next starts
    for name, lr in LR.items():
        if lr is None or (name == "nested_d" and D_LAM is None):
            continue
        kw = dict(rule_kwargs=dict(surprise_weight=D_LAM)) if name == "nested_d" else {}
        CONTROL.append(sr.arm_cfg(name, stage="stage1_nested_control", epochs=EPOCHS, seed=s, lr=lr,
                                  eval_split="test", save_checkpoints=True, **kw))
for name, lr in LR.items():
    if lr is None or (name == "nested_d" and D_LAM is None): print(f"skipping {name}: set LR / D_LAM first")
for c in CONTROL: print(run_id(c), sr.arm_label(c), c["lr"], c["seed"])

In [ ]:
out = run_sweep(CONTROL, sr.run_experiment, paths)

In [ ]:
OLD_CONTROL = [sr.make_cfg(stage="stage1_control", strategy="naive", epochs=EPOCHS, seed=s, lr=lr, eval_split="test",
                           save_checkpoints=True, **arm)
               for s in SEEDS for lr, arm in [(0.1, dict(lr_schedule="flat")),
                                              (0.6, dict(lr_schedule="llrd", level_ratios=(1, 4, 16), level_order="fast_deep"))]]
cdf = table(OLD_CONTROL + CONTROL)
done = cdf[cdf["status"] != "not finished"] if "status" in cdf else cdf
assert done["steps"].nunique() == 1 and done["n_params"].nunique() == 1, "steps or param count differ between arms. The comparison is NOT matched"
cols = ["avg_incremental_acc", "acc_just_learned", "final_avg_acc", "avg_forgetting", "minutes", "peak_gpu_mb"]
display(done.groupby(["arm", "lr"])[cols].agg(["mean", "std", "count"]).round(2))
lvl = [c for c in done.columns if c.startswith(("updates/", "rel_displacement/", "penalty_ratio/"))]
if lvl: display(done.groupby(["arm", "lr"])[lvl].mean().T.round(4))

**Next:** run `stage1_analysis`, `stage1_drift_diagnostic` and `stage1_structural` (CPU runtimes). They pick up both
`stage1_control` and `stage1_nested_control`, so old and new arms land in one table. Gate 1 (proposed) is judged on
per-level probe forgetting at matched just-learned accuracy, against Flat at its tuned lr and LLRD.

## 5. D: learning-rate re-check at the chosen λ (validation, seed 0)

D's lr (0.7) was picked at λ = 0.1, where the penalty was 5-10× the data gradient and the clip shrank the data part of
each step, so the best lr may have moved at λ = 0.01. Same selection rule (highest `avg_incremental_acc`).
The lr 0.7 point already exists from the λ grid (same config, reused). If the best lr here differs clearly
(beyond ~0.5 points of noise), re-run D's control at that lr and report both; otherwise keep 0.7.

In [ ]:
D_LAM_CHOSEN = 0.01
D_LR_RECHECK = [0.2, 0.4, 0.7]
D_RECHECK = [val_cfg("nested_d", lr, rule_kwargs=dict(surprise_weight=D_LAM_CHOSEN)) for lr in D_LR_RECHECK]
for c in D_RECHECK: print(run_id(c), sr.arm_label(c), c["lr"])
out = run_sweep(D_RECHECK, sr.run_experiment, paths)
rdf = table(D_RECHECK)
display(rdf.drop(columns=["seed", "run_id"], errors="ignore"))
done = rdf.dropna(subset=["avg_incremental_acc"]) if "avg_incremental_acc" in rdf else rdf.iloc[:0]
if len(done):
    best = done.loc[done["avg_incremental_acc"].idxmax()]
    print(f"best D lr at lambda {D_LAM_CHOSEN}: {best['lr']} (avg incremental {best['avg_incremental_acc']:.2f})")
    if best["lr"] in (min(D_LR_RECHECK), max(D_LR_RECHECK)): print("WARNING: best lr is at the edge of the re-check grid.")

## 6. Steps per task: Flat at 2 and 3 epochs per task (test set, 3 seeds, checkpoints) — plan R5

**Question:** is deep-layer drift set by the *number* of steps per task rather than the step size? So far a level's own
drift was the same (layer3 ≈ 4.7-4.9 points) whatever its lr, schedule or stepping period, which fits the hypothesis
that ~790 steps per task let every level converge regardless. With 2-3 epochs there are ~160-240 steps per task.

**Predictions, written before running:** if the hypothesis holds, own drift (drift diagnostic) and probe forgetting at
layer3/layer4 drop clearly at 2-3 epochs compared with 10. If they stay about the same, the number of steps is not the
limit and the hypothesis is wrong.

**Confound to keep in mind:** fewer epochs also means each task is learned less well (lower `acc_just_learned`), and
probe forgetting is measured from that lower starting point. Always read it next to `acc_just_learned` and
`probe_at_learn`; a drop in forgetting that only mirrors a lower peak is not protection.

lr stays at Flat's tuned 0.1 (tuned at 10 epochs; it is not re-tuned here, since this is a diagnostic, not an arm).
Flat runs through the same path as B and D. Optional: add `"nested_b"` to `EPT_ARMS` (at its lr) to test whether slowing
levels starts to help once there are fewer steps per task — that is the comparison that would matter for nesting.

In [ ]:
EPT_EPOCHS = [2, 3]
EPT_ARMS = {"flat": 0.1}          # optional: "nested_b": 0.6
EPT = [sr.arm_cfg(name, stage="stage1_epochs_per_task", epochs=ep, seed=s, lr=lr,
                  eval_split="test", save_checkpoints=True)
       for s in SEEDS for ep in EPT_EPOCHS for name, lr in EPT_ARMS.items()]
for c in EPT: print(run_id(c), sr.arm_label(c), "epochs", c["epochs"], "seed", c["seed"])
out = run_sweep(EPT, sr.run_experiment, paths)

In [ ]:
REF = [c for c in CONTROL if sr.arm_label(c) in {sr.arm_label(sr.arm_cfg(n, epochs=EPOCHS)) for n in EPT_ARMS}]   # same arms at 10 epochs
edf = table(REF + EPT)
edf["epochs"] = [c["epochs"] for c in REF + EPT]
cols = ["avg_incremental_acc", "acc_just_learned", "final_avg_acc", "minutes"]
display(edf.groupby(["arm", "epochs"])[cols].agg(["mean", "std", "count"]).round(2))
lvl = [c for c in edf.columns if c.startswith("rel_displacement/")]
display(edf.groupby(["arm", "epochs"])[lvl].mean().T.round(4))

Then run `stage1_analysis` and `stage1_drift_diagnostic` (they include `stage1_epochs_per_task`): compare probe forgetting
and **own** drift per level at 2 / 3 / 10 epochs, next to `probe_at_learn` / `acc_just_learned`.